In [ ]:
!pip install -q geoip2 lightgbm xgboost scikit-learn pandas numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import geoip2.database, ipaddress
geo_reader=geoip2.database.Reader('/content/drive/MyDrive/CICIDS2017/GeoLite2-City.mmdb')
def get_country(ip):
    try:
        if ipaddress.ip_address(ip).is_private: return 'Not Geolocatable (Private/Internal IP)'
        return geo_reader.city(ip).country.name or 'Unknown'
    except Exception: return 'Not Geolocatable (Invalid IP)'
print(get_country('8.8.8.8'))

Mounted at /content/drive
United States


## 1. Data (same sampling and filtering as the paper) + host-behaviour features

In [ ]:
import os, gc, subprocess, numpy as np, pandas as pd
SEED=42; DATA_DIR='/content'
scenarios={42:"https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-42/capture20110810.binetflow.2format",
43:"https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-43/capture20110811.binetflow.2format",
44:"https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-44/capture20110812.binetflow.2format"}
cols=['saddr','daddr','proto','sport','dport','state','stos','dtos','swin','dwin','shops','dhops','stime','ltime','sttl','dttl','tcprtt','synack','ackdat','spkts','dpkts','sbytes','dbytes','sappbytes','dappbytes','dur','pkts','bytes','appbytes','rate','srate','drate','label']
keep=['saddr','daddr','proto','dport','state','sttl','dttl','spkts','dpkts','sbytes','dbytes','dur','rate','srate','drate','label']
dfs=[]
for sid,url in scenarios.items():
    f=f'{DATA_DIR}/ctu13_scenario{sid}.binetflow'
    if not os.path.exists(f) or os.path.getsize(f)==0: subprocess.run(['wget','-q','-O',f,url])
    parts=[]
    for ch in pd.read_csv(f,names=cols,usecols=keep,low_memory=False,chunksize=200000):
        ch=ch[ch['saddr']!='SrcAddr']
        parts.append(ch.sample(frac=0.3,random_state=42) if len(ch)>50000 else ch)
    d=pd.concat(parts,ignore_index=True); d['scenario']=sid; dfs.append(d); del parts; gc.collect()
df=pd.concat(dfs,ignore_index=True); del dfs; gc.collect()
df['Label']=df['label'].astype(str).str.lower().apply(lambda x:1 if 'botnet' in x else 0)
num=['dur','spkts','dpkts','sbytes','dbytes','rate','srate','drate','sttl','dttl']
for c in num: df[c]=pd.to_numeric(df[c],errors='coerce')
# ---- host-behaviour features: per (scenario, source IP), NO labels used ----
g=df.groupby(['scenario','saddr'])
host=pd.DataFrame({'h_flows':g.size(),'h_uniq_dst':g['daddr'].nunique(),'h_uniq_dport':g['dport'].nunique(),
    'h_mean_dur':g['dur'].mean(),'h_mean_sbytes':g['sbytes'].mean(),'h_std_sbytes':g['sbytes'].std().fillna(0),
    'h_mean_spkts':g['spkts'].mean()})
host['h_dst_per_flow']=host['h_uniq_dst']/host['h_flows']; host['h_dport_per_flow']=host['h_uniq_dport']/host['h_flows']
df=df.join(host,on=['scenario','saddr']); hcols=list(host.columns); del host,g; gc.collect()
df=df.dropna(subset=num)
ts=df['state'].value_counts().head(10).index; df['state_g']=df['state'].where(df['state'].isin(ts),'Other')
tp=df['proto'].value_counts().head(5).index; df['proto_g']=df['proto'].where(df['proto'].isin(tp),'Other')
df=pd.concat([df,pd.get_dummies(df['proto_g'],prefix='proto'),pd.get_dummies(df['state_g'],prefix='state')],axis=1)
flow_cols=num+[c for c in df.columns if (c.startswith('proto_') or c.startswith('state_')) and c not in ['proto_g','state_g']]
host_cols=flow_cols+hcols
X=df[host_cols].astype('float32'); y=df['Label'].values; meta=df[['saddr','daddr']].reset_index(drop=True)
X=X.reset_index(drop=True); del df; gc.collect()
print(X.shape,'attacks:',int(y.sum()),'| flow feats',len(flow_cols),'| +host feats',len(host_cols))

(2516046, 36) attacks: 11403 | flow feats 27 | +host feats 36


## 2. Helpers

In [ ]:
from sklearn.metrics import precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,precision_recall_curve
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
def metrics(yt,p,s): return dict(Precision=precision_score(yt,p,zero_division=0),Recall=recall_score(yt,p),F1=f1_score(yt,p),ROC_AUC=roc_auc_score(yt,s),PR_AUC=average_precision_score(yt,s))
def best_thr(yt,s):
    p,r,t=precision_recall_curve(yt,s); f=2*p[:-1]*r[:-1]/np.clip(p[:-1]+r[:-1],1e-12,None); return float(t[np.argmax(f)])
def logit(p): p=np.clip(p,1e-6,1-1e-6); return np.log(p/(1-p))
_cc={}; _vc={}
def vip(ip):
    try: ipaddress.ip_address(str(ip)); return True
    except ValueError: return False
def geo_signals(idx):
    fr=meta.iloc[idx];
    for ip in pd.unique(pd.concat([fr['saddr'],fr['daddr']])):
        if ip not in _cc: _cc[ip]=get_country(ip); _vc[ip]=vip(ip)
    s=fr['saddr'].map(_cc); d=fr['daddr'].map(_cc)
    ok=~(s.str.startswith('Not Geolocatable')|d.str.startswith('Not Geolocatable')|(s=='Unknown')|(d=='Unknown'))
    return ((s!=d)&ok).astype(float).values,(fr['saddr'].map(_vc)&fr['daddr'].map(_vc)).astype(float).values
def mcnemar(yt,p1,p2):
    from scipy.stats import binomtest
    a=(p1==yt); b=(p2==yt); n1=int((a&~b).sum()); n2=int((~a&b).sum()); return (1.0 if n1+n2==0 else float(binomtest(min(n1,n2),n1+n2,0.5).pvalue)),n1,n2

## 3. Tune hyperparameters ONCE on seed 42 (validation only; same budget for XGBoost and LightGBM, flow-only and +host)

In [ ]:
idx=np.arange(len(y))
tr42,te42=train_test_split(idx,test_size=0.2,stratify=y,random_state=42)
f42,v42=train_test_split(tr42,test_size=0.25,stratify=y[tr42],random_state=SEED)
pos_w=float((y[f42]==0).sum()/max(1,(y[f42]==1).sum())); rng=np.random.default_rng(SEED); N_TRIALS=6
def sample_xgb(): return dict(max_depth=int(rng.integers(4,11)),learning_rate=float(rng.choice([0.05,0.1,0.2])),subsample=float(rng.choice([0.7,0.85,1.0])),colsample_bytree=float(rng.choice([0.6,0.8,1.0])),min_child_weight=int(rng.choice([1,3,5])),scale_pos_weight=float(rng.choice([1,np.sqrt(pos_w),pos_w/4])))
def sample_lgb(): return dict(num_leaves=int(rng.choice([31,63,127,255])),learning_rate=float(rng.choice([0.05,0.1,0.2])),subsample=float(rng.choice([0.7,0.85,1.0])),subsample_freq=1,colsample_bytree=float(rng.choice([0.6,0.8,1.0])),min_child_samples=int(rng.choice([10,20,50])),scale_pos_weight=float(rng.choice([1,np.sqrt(pos_w),pos_w/4])))
def fit_xgb(pr,Xa,ya,Xb,yb):
    m=XGBClassifier(n_estimators=600,early_stopping_rounds=30,random_state=SEED,n_jobs=-1,eval_metric='aucpr',tree_method='hist',**pr); m.fit(Xa,ya,eval_set=[(Xb,yb)],verbose=False); return m
def fit_lgb(pr,Xa,ya,Xb,yb):
    m=LGBMClassifier(n_estimators=600,random_state=SEED,n_jobs=-1,verbose=-1,**pr); m.fit(Xa,ya,eval_set=[(Xb,yb)],eval_metric='average_precision',callbacks=[early_stopping(30,verbose=False)]); return m
def tune(sampler,fit,cols):
    Xa,Xb=X.iloc[f42][cols],X.iloc[v42][cols]; best=(-1,None)
    for t in range(N_TRIALS):
        pr=sampler(); m=fit(pr,Xa,y[f42],Xb,y[v42]); s=average_precision_score(y[v42],m.predict_proba(Xb)[:,1]); print(round(s,4),pr)
        if s>best[0]: best=(s,pr)
    return best[1]
HP={}
for nm,sm,ft,cl in [('xgb_flow',sample_xgb,fit_xgb,flow_cols),('lgb_flow',sample_lgb,fit_lgb,flow_cols),('xgb_host',sample_xgb,fit_xgb,host_cols),('lgb_host',sample_lgb,fit_lgb,host_cols)]:
    print('tuning',nm); HP[nm]=tune(sm,ft,cl); gc.collect()
print(HP)

tuning xgb_flow
0.9531 {'max_depth': 4, 'learning_rate': 0.2, 'subsample': 0.85, 'colsample_bytree': 0.8, 'min_child_weight': 3, 'scale_pos_weight': 54.91029669687226}
0.9489 {'max_depth': 4, 'learning_rate': 0.2, 'subsample': 0.7, 'colsample_bytree': 0.6, 'min_child_weight': 3, 'scale_pos_weight': 54.91029669687226}
0.9592 {'max_depth': 9, 'learning_rate': 0.2, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'scale_pos_weight': 1.0}
0.9637 {'max_depth': 9, 'learning_rate': 0.1, 'subsample': 0.85, 'colsample_bytree': 0.8, 'min_child_weight': 1, 'scale_pos_weight': 54.91029669687226}
0.9627 {'max_depth': 9, 'learning_rate': 0.1, 'subsample': 0.85, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'scale_pos_weight': 14.820296447355195}
0.9553 {'max_depth': 7, 'learning_rate': 0.05, 'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'scale_pos_weight': 1.0}
tuning lgb_flow
0.6573 {'num_leaves': 255, 'learning_rate': 0.2, 'subsample': 0.7, 'subsample_freq': 

## 4. Five seeds: train, learn AGEF-Net weights on validation, evaluate on test once

In [ ]:
SYS=['LightGBM flow-only','XGBoost flow-only','LightGBM +host','XGBoost +host','AGEF-Net (flow-only signals)','AGEF-Net v4 (+host signals)']
allrows=[]; keep42={}
for seed in [42,43,44,45,46]:
    tr,te=train_test_split(idx,test_size=0.2,stratify=y,random_state=seed)
    fi,vi=train_test_split(tr,test_size=0.25,stratify=y[tr],random_state=SEED)
    sc_v={};sc_t={}
    for nm,ft,cl in [('xgb_flow',fit_xgb,flow_cols),('lgb_flow',fit_lgb,flow_cols),('xgb_host',fit_xgb,host_cols),('lgb_host',fit_lgb,host_cols)]:
        m=ft(HP[nm],X.iloc[fi][cl],y[fi],X.iloc[vi][cl],y[vi]); sc_v[nm]=m.predict_proba(X.iloc[vi][cl])[:,1]; sc_t[nm]=m.predict_proba(X.iloc[te][cl])[:,1]; del m; gc.collect()
    Gv,Ev=geo_signals(vi); Gt,Et=geo_signals(te)
    def agef(cn,dn):
        Zv=np.c_[logit(sc_v[cn]),logit(sc_v[dn]),Gv,Ev]; Zt=np.c_[logit(sc_t[cn]),logit(sc_t[dn]),Gt,Et]
        st=LogisticRegression(C=1.0,max_iter=2000,random_state=SEED)
        oof=cross_val_predict(st,Zv,y[vi],cv=StratifiedKFold(5,shuffle=True,random_state=SEED),method='predict_proba')[:,1]
        st.fit(Zv,y[vi]); return oof,st.predict_proba(Zt)[:,1],st.coef_[0]
    a_f=agef('xgb_flow','lgb_flow'); a_h=agef('xgb_host','lgb_host')
    systems={'LightGBM flow-only':(sc_v['lgb_flow'],sc_t['lgb_flow']),'XGBoost flow-only':(sc_v['xgb_flow'],sc_t['xgb_flow']),
             'LightGBM +host':(sc_v['lgb_host'],sc_t['lgb_host']),'XGBoost +host':(sc_v['xgb_host'],sc_t['xgb_host']),
             'AGEF-Net (flow-only signals)':a_f[:2],'AGEF-Net v4 (+host signals)':a_h[:2]}
    for n,(sv,st_) in systems.items():
        th=best_thr(y[vi],sv); p=(st_>=th).astype(int)
        allrows.append({'seed':seed,'System':n,**metrics(y[te],p,st_)})
        if seed==42: keep42[n]=p
    if seed==42: y42=y[te]; print('seed 42 learned weights [logit C, logit D, G, E] (+host):',np.round(a_h[2],3))
    print('seed',seed,'done'); gc.collect()
R=pd.DataFrame(allrows); R.to_csv('table_v4_all_seeds.csv',index=False)
summ=R.groupby('System')[['Precision','Recall','F1','ROC_AUC','PR_AUC']].agg(['mean','std']).round(4).reindex(SYS)
print(summ.to_string()); summ.to_csv('table_v4_mean_sd.csv')

seed 42 learned weights [logit C, logit D, G, E] (+host): [ 0.917  0.884 -0.071 -0.151]
seed 42 done
seed 43 done
seed 44 done
seed 45 done
seed 46 done
                             Precision          Recall              F1         ROC_AUC          PR_AUC        
                                  mean     std    mean     std    mean     std    mean     std    mean     std
System                                                                                                        
LightGBM flow-only              0.9600  0.0106  0.8731  0.0123  0.9144  0.0042  0.9995  0.0001  0.9601  0.0026
XGBoost flow-only               0.9640  0.0094  0.8764  0.0108  0.9180  0.0023  0.9996  0.0001  0.9656  0.0023
LightGBM +host                  0.9996  0.0008  0.9999  0.0002  0.9998  0.0004  1.0000  0.0000  0.9996  0.0009
XGBoost +host                   1.0000  0.0000  0.9999  0.0002  1.0000  0.0001  1.0000  0.0000  1.0000  0.0000
AGEF-Net (flow-only signals)    0.9630  0.0054  0.8793  0.0051  0.9192

## 5. Paired comparison and honest verdict

In [ ]:
piv=R.pivot(index='seed',columns='System',values='F1')
ag='AGEF-Net v4 (+host signals)'
for b in ['XGBoost +host','XGBoost flow-only','LightGBM +host','LightGBM flow-only']:
    dlt=piv[ag]-piv[b]; print(f'F1 AGEF-v4 minus {b}: mean {dlt.mean():+.4f} (sd {dlt.std():.4f}); AGEF higher on {(dlt>0).sum()}/5 seeds')
for b in ['XGBoost +host','XGBoost flow-only']:
    p,n1,n2=mcnemar(y42,keep42[b],keep42[ag]); print(f'McNemar seed42 {b} vs AGEF-v4: p={p:.4g} (baseline-only-correct={n1}, AGEF-only-correct={n2})')
print('\n--- Mean over 5 seeds, AGEF-v4 vs best baseline per metric ---')
m=R.groupby('System')[['Precision','Recall','F1','ROC_AUC','PR_AUC']].mean()
for k in m.columns:
    bb=m.drop(index=[ag,'AGEF-Net (flow-only signals)'])[k]; print(f'{k:10s} AGEF-v4={m.loc[ag,k]:.4f}  best baseline={bb.max():.4f} ({bb.idxmax()})')
print('\nReport this table exactly. If XGBoost +host matches AGEF-v4, the gain comes from the host features, and the paper should say so.')
import shutil,glob
os.makedirs('/content/drive/MyDrive/agef_backup',exist_ok=True)
for f in glob.glob('/content/table_v4_*.csv'): shutil.copy(f,'/content/drive/MyDrive/agef_backup/')

F1 AGEF-v4 minus XGBoost +host: mean -0.0002 (sd 0.0005); AGEF higher on 0/5 seeds
F1 AGEF-v4 minus XGBoost flow-only: mean +0.0817 (sd 0.0025); AGEF higher on 5/5 seeds
F1 AGEF-v4 minus LightGBM +host: mean -0.0000 (sd 0.0006); AGEF higher on 1/5 seeds
F1 AGEF-v4 minus LightGBM flow-only: mean +0.0853 (sd 0.0044); AGEF higher on 5/5 seeds
McNemar seed42 XGBoost +host vs AGEF-v4: p=1 (baseline-only-correct=0, AGEF-only-correct=0)
McNemar seed42 XGBoost flow-only vs AGEF-v4: p=5.322e-110 (baseline-only-correct=0, AGEF-only-correct=364)

--- Mean over 5 seeds, AGEF-v4 vs best baseline per metric ---
Precision  AGEF-v4=1.0000  best baseline=1.0000 (XGBoost +host)
Recall     AGEF-v4=0.9995  best baseline=0.9999 (LightGBM +host)
F1         AGEF-v4=0.9997  best baseline=1.0000 (XGBoost +host)
ROC_AUC    AGEF-v4=1.0000  best baseline=1.0000 (XGBoost +host)
PR_AUC     AGEF-v4=1.0000  best baseline=1.0000 (XGBoost +host)

Report this table exactly. If XGBoost +host matches AGEF-v4, the gain com